#### Load in everything.

In [17]:
!pip install scikit-learn

In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
# Modelling
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
df = pd.read_csv('../data/carcsvdata/cleaned_car_data.csv')

In [19]:
df.head()

,Make,Price,Year,Kilometer,Fuel Type,Transmission,Location,Color,Owner,Seller Type,Drivetrain,Seating Capacity,Fuel Tank Capacity,Volume,Power_bhp,Power_rpm,Torque_Nm,Torque_rpm
0,Honda,505000,2017,87150,Petrol,Manual,Pune,Grey,First,Corporate,FWD,5.0,35.0,1.008832e+07,87.0,6000.0,109.0000,4500.0
1,Maruti Suzuki,450000,2014,75000,Diesel,Manual,Ludhiana,White,Second,Individual,FWD,5.0,42.0,1.052972e+07,74.0,4000.0,190.0000,2000.0
2,Hyundai,220000,2011,67000,Petrol,Manual,Lucknow,Maroon,First,Individual,FWD,5.0,35.0,8.863016e+06,79.0,6000.0,112.7619,4000.0
3,Toyota,799000,2019,37500,Petrol,Manual,Mangalore,Red,First,Individual,FWD,5.0,37.0,1.052663e+07,82.0,6000.0,113.0000,4200.0
4,Toyota,1950000,2018,69000,Diesel,Manual,Mumbai,Grey,First,Individual,RWD,7.0,55.0,1.555376e+07,148.0,3400.0,343.0000,1400.0


#### Preparing X and y values.

In [20]:
y = df['Price']
X = df.drop(columns=['Price'])

In [23]:
# Create Column Transformer with 3 types of transformers
num_features = X.select_dtypes(exclude="object").columns
cat_features = X.select_dtypes(include="object").columns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", oh_transformer, cat_features),
         ("StandardScaler", numeric_transformer, num_features),        
    ]
)

C:\Users\amito\AppData\Local\Temp\ipykernel_12840\2177192774.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = X.select_dtypes(include="object").columns


#### Train/Test split and transforming the data. 

In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [25]:
X_train.shape, X_test.shape

((1441, 145), (361, 145))

#### Create an Evaluate Function to give all metrics after model Training

In [26]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

In [28]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "AdaBoost Regressor": AdaBoostRegressor()
}
model_list = []
r2_list =[]

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Evaluate Train and Test dataset
    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    
    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    r2_list.append(model_test_r2)
    
    print('='*35)
    print('\n')

Linear Regression
Model performance for Training set
- Root Mean Squared Error: 1058073.7330
- Mean Absolute Error: 581233.7001
- R2 Score: 0.8201
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 1295033.9412
- Mean Absolute Error: 698257.5853
- R2 Score: 0.6831


Lasso
Model performance for Training set
- Root Mean Squared Error: 1058073.7514
- Mean Absolute Error: 581231.2172
- R2 Score: 0.8201
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 1292927.6133
- Mean Absolute Error: 694848.9983
- R2 Score: 0.6842


Ridge
Model performance for Training set
- Root Mean Squared Error: 1115794.0859
- Mean Absolute Error: 617727.0176
- R2 Score: 0.7999
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 1303406.2025
- Mean Absolute Error: 721595.3959
- R2 Score: 0.6790


K-Neighbors Regressor
Model performance for Training set
- Root Mean Squared Error: 957966.2335
-

c:\amitojpML1\venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:784: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.952727e+14, tolerance: 8.966e+11
  model = cd_fast.sparse_enet_coordinate_descent(


Decision Tree
Model performance for Training set
- Root Mean Squared Error: 543.0783
- Mean Absolute Error: 27.7585
- R2 Score: 1.0000
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 1297259.1375
- Mean Absolute Error: 425825.4903
- R2 Score: 0.6821


Random Forest Regressor
Model performance for Training set
- Root Mean Squared Error: 350368.5959
- Mean Absolute Error: 102351.0227
- R2 Score: 0.9803
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 992617.9600
- Mean Absolute Error: 323657.1072
- R2 Score: 0.8138


AdaBoost Regressor
Model performance for Training set
- Root Mean Squared Error: 1565330.9894
- Mean Absolute Error: 1462634.9364
- R2 Score: 0.6062
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 1848759.5125
- Mean Absolute Error: 1589126.6734
- R2 Score: 0.3543




In [30]:
import pandas as pd
results = pd.DataFrame({'Model': model_list, 'Test R2': r2_list}).sort_values(by='Test R2', ascending=False)
print(results)

                     Model   Test R2
5  Random Forest Regressor  0.813849
3    K-Neighbors Regressor  0.694900
1                    Lasso  0.684172
0        Linear Regression  0.683142
4            Decision Tree  0.682052
2                    Ridge  0.679032
6       AdaBoost Regressor  0.354252


In [31]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor

# Parameter distribution to sample from
param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [None, 5, 10, 15, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4, 6],
    'max_features': [1.0, 'sqrt', 'log2', 0.3, 0.5],
    'bootstrap': [True, False]
}

rf = RandomForestRegressor(random_state=42)

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=50,          # number of parameter combinations to try
    cv=5,               # 5-fold cross-validation
    scoring='r2',
    verbose=2,
    random_state=42,
    n_jobs=-1           # use all available cores
)

random_search.fit(X_train, y_train)

print("Best parameters found:", random_search.best_params_)
print("Best CV R2 score:", random_search.best_score_)

# Evaluate the tuned model on train/test
best_rf = random_search.best_estimator_

train_r2 = best_rf.score(X_train, y_train)
test_r2 = best_rf.score(X_test, y_test)

print(f"Tuned Random Forest - Train R2: {train_r2:.4f}")
print(f"Tuned Random Forest - Test R2: {test_r2:.4f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best parameters found: {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 0.3, 'max_depth': None, 'bootstrap': False}
Best CV R2 score: 0.8515607937970089
Tuned Random Forest - Train R2: 0.9716
Tuned Random Forest - Test R2: 0.8030


In [33]:
import joblib
joblib.dump(best_rf, '../models/random_forest_model.pkl')
joblib.dump(preprocessor, '../models/preprocessor.pkl')

['../models/preprocessor.pkl']